# Laya PoC - Phase 4: the baselines (Colab G4 from VS Code)

Design doc §6.2 Phase 4, §5.13 and §7.10: every baseline gets predictions for every split, in the same run layout, `results/runs.csv` and report as the Phase 3 matrix (B2, zero-shot Laya, already ran there). B3 (TF-IDF + LR) and B4 (fine-tuned small encoders, 3 seeds each) are the references of decision criterion 1: fine-tuned Laya must beat both by at least 3 macro-F1 points out of distribution. Each run fits on the clean train split (B4 on the same augmented epochs Laya saw), gets its temperature on `val`, and is evaluated on every split (val, test_id, ood_country, ood_script, ood_brand, stripped_test, trap_candidates) with per-row predictions under `runs/p4/<run>/preds/`. Runbook: `docs/phase4_baselines_runbook.md`.

| Step | Arm | What | Runs | Estimate (G4) |
|---|---|---|---|---|
| 7 | B1 | majority class + class prior (CPU), c10/c7 | 4 | ~2-3 min |
| 8 | B3 | char TF-IDF + logistic regression, C and T on val (CPU), c10/c7 | 2 | ~3-5 min |
| 9 | B4 | fine-tuned small encoder, `modernbert_base` + `mmbert_small`, c10, seeds 11, 22, 33 | 6 | ~20-35 min |
| 10 | B5 | zero-shot LLM reference, scored subsets, `qwen3_4b`, c10, optional (flag `RUN_B5`) | 1 | ~10-20 min |

13 runs. **Expected runtime:** about 1-1.5 h in total: setup and the frozen data 10-15 min, variants 2-5 min, then the arms above (estimates, see the runbook).

**How to connect:** **Select Kernel → Colab → New Colab Server → GPU → G4**, then the Python 3 kernel. Do
**not** use *Auto Connect* (it gives a CPU). The G4 uses paid compute units: check Colab's Resources panel.
If the extension assigns another GPU, Step 4 warns (B4 and B5 then take longer). Keep VS Code open and the laptop awake: a running cell keeps the server alive. The Hugging Face token is asked in a masked input box (Step 3) and never stored in this notebook.

**After a kernel restart or a Colab disconnect:** re-run **Step 1**, then the interrupted step (or **Run All**). Finished steps skip themselves (their outputs are on disk) and the matrix skips finished runs (`done.json`); an unfinished baseline run starts again (B4 runs take a few minutes each and are not resumable). `REDO = True` in Step 1 redoes Steps 4-6 but never a finished run: delete `runs/p4/<run_name>` to redo one. If the server itself was removed, `/content` is gone: start again from Step 1.

**Phase 4 exit check** (Step 11, design §6.2): every configured baseline run has predictions for every eval split (B5: its subsets; B5 switched off is not a failure). The report also gives an early read of decision criterion 1 once it is merged with the Phase 3 runs locally (runbook).

Code bundle: 86 files, sha256 `cf73c341309ed89673b8e1f7ab84bfb570e9da8b0e90383137fdb2ff3bf7f1d2` (verified when Step 1 unpacks it).


### Step 1 - Setup
Non-interactive environment, work dir, code bundle, helpers, and `CARD` (default `G4`, config `phase4.card`). Seconds. **Re-run this cell first after any kernel restart or Colab disconnect.**

In [ ]:
# Step 1 - Setup
import getpass, json, os, shutil, signal, subprocess, sys, time, zipfile
from pathlib import Path

os.environ.update({
    "COLAB_DISABLE_STDIN_FOR_SHELL_MAGICS": "1",  # a command that prompts gets EOF instead of hanging
    "GIT_TERMINAL_PROMPT": "0", "PIP_NO_INPUT": "1", "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TOKENIZERS_PARALLELISM": "false",
    "LAYA_CUDA_AMP": "fp16",  # fp16 autocast on every card, also if the extension assigns an L4/A100
})
WORK = Path("/content/laya_poc")
DATA, RUNS = WORK / "data", WORK / "runs" / "p4"
LOGS = RUNS / "logs"
for _d in (DATA, LOGS):
    _d.mkdir(parents=True, exist_ok=True)
os.environ["LAYA_POC_ROOT"] = str(WORK)  # the CLIs read WORK/config.yaml
REDO = False  # True: recompute steps whose outputs already exist
# The card this session runs on (config phase4.card), recorded with every run in results/runs.csv. The estimates
# assume a G4; B1 and B3 run on the CPU whatever the card.
CARD = "G4"
RESULTS = WORK / "results"  # results/runs.csv, phase4_report.md (+ .json), written by the matrix
DRILL_MARK = Path.home() / ".laya_poc_kill_drill.json"  # outside WORK: Step 13d can tell a wiped /content


def write_json(path, obj) -> None:
    """Write JSON atomically, so a kill mid-write never leaves half a file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

def run_logged(cmd, log_path, expect_returncode=0, hint=None) -> int:
    """Run cmd with stdin closed (nothing can prompt); stream its output here and into log_path.

    VS Code truncates long cell output, so the log file is the full record. expect_returncode is an
    int, a tuple of ints, or None (accept any). Raises on an unexpected code; returns the code.
    """
    cmd, log_path = [str(c) for c in cmd], Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8", buffering=1) as log:
        log.write("$ " + " ".join(cmd) + "\n")
        with subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1, env=os.environ.copy()) as proc:
            try:
                for line in proc.stdout:
                    log.write(line)
                    print(line, end="", flush=True)
            except BaseException:  # interrupted cell: never leave an orphan holding the GPU
                proc.kill()
                raise
    rc = proc.returncode
    expected = (expect_returncode,) if isinstance(expect_returncode, int) else expect_returncode
    if expected is not None and rc not in expected:
        name = cmd[cmd.index("-m") + 1] if "-m" in cmd else Path(cmd[0]).name
        msg = f"{name} exited with {rc} (expected {' or '.join(map(str, expected))}); full log: {log_path}"
        raise RuntimeError(msg + (f"\nhint: {hint}" if hint else ""))
    return rc

def sh(cmd, check=True) -> str:
    """Run a short command with stdin closed; print and return its combined output."""
    res = subprocess.run(cmd, shell=isinstance(cmd, str), stdin=subprocess.DEVNULL, capture_output=True,
                         text=True, errors="replace")
    out = (res.stdout + res.stderr).strip()
    if out:
        print(out)
    if check and res.returncode:
        raise RuntimeError(f"command failed with exit code {res.returncode}: {cmd}")
    return out

def already_done(path) -> bool:
    """True (and says so) when a step's output exists and REDO is False, so re-runs skip finished steps."""
    if Path(path).exists() and not globals().get("REDO", False):
        print(f"skip: {path} exists (set REDO = True in Step 1 to recompute)")
        return True
    return False

def fresh_dir(path) -> Path:
    """Empty a run directory so a re-run starts from scratch instead of resuming a stale checkpoint."""
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path

def show_json(path, keys=None, width=150):
    """Print one line per top-level key (only `keys` when any of them exist), without bulky per-row or
    per-class detail; return the whole object."""
    def brief(v):
        if isinstance(v, dict):
            return {k: brief(x) for k, x in v.items() if k not in ("per_row", "per_class", "cm", "confusion")}
        return v
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    shown = brief(obj) if isinstance(obj, dict) else {"value": obj}
    if keys and any(k in shown for k in keys):
        shown = {k: shown[k] for k in keys if k in shown}
    for key, val in shown.items():
        text = json.dumps(val, ensure_ascii=False, separators=(",", ":"))
        print(f"  {key}: {text if len(text) <= width else text[:width - 3] + '...'}")
    return obj

def show_markdown(path) -> None:
    """Render a markdown file in the notebook (plain text outside IPython, e.g. in the local dry run)."""
    text = Path(path).read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(text)
        return
    display(Markdown(text))

def restore_hf_token() -> bool:
    """Put a previously saved HF token into os.environ for the CLIs. Never prints it."""
    if os.environ.get("HF_TOKEN"):
        return True
    try:
        from huggingface_hub import constants  # importing constants does not query the Colab vault
        token_file = Path(constants.HF_TOKEN_PATH)
    except ImportError:
        token_file = Path.home() / ".cache" / "huggingface" / "token"
    token = token_file.read_text(encoding="utf-8").strip() if token_file.exists() else ""
    if token:
        os.environ["HF_TOKEN"] = token
    return bool(token)

def resume_blocker(runs, kill_at=None):
    """Why Step 10 must not run now, or None. A resume is only valid straight after the crash run: the
    resumed log must end at the injected crash (at micro-step kill_at, when given). A -9 without it is some
    other kill, e.g. the host OOM killer. A second resume (interrupted Step 10, or REDO) would restart from
    a later checkpoint and the report, which grades the last 'resumed' event, would silently FAIL."""
    runs = Path(runs)
    exit_file, log_file = runs / "crash_exit.json", runs / "resumed" / "log.jsonl"
    if not exit_file.exists():
        return "no crash_exit.json: run Step 9 (crash run) first"
    rc = json.loads(exit_file.read_text(encoding="utf-8")).get("returncode")
    if rc not in (-9, 137):
        return f"the crash run exited with {rc}, not -9/137 (hard kill)"
    events = []
    for line in log_file.read_text(encoding="utf-8").splitlines() if log_file.exists() else []:
        try:
            event = json.loads(line)
        except ValueError:  # a SIGKILL can leave half a line
            continue
        if isinstance(event, dict):
            events.append(event)
    names = [e.get("event") for e in events]
    if "crash_injected" not in names:
        return "resumed/log.jsonl has no crash_injected event"
    if names[-1] != "crash_injected":
        return "a resume already ran (or was interrupted) after the crash"
    step = events[-1].get("micro_step")
    if kill_at is not None and step != kill_at:
        return f"the crash was injected at micro-step {step}, not at the planned {kill_at}"
    return None

def log_tail(path, n=8) -> str:
    """The last n lines of a log file, for error messages ('' when it does not exist)."""
    path = Path(path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-n:])

def gpu_pids() -> list[int]:
    """PIDs of processes holding GPU memory according to nvidia-smi; [] without an NVIDIA driver."""
    if not shutil.which("nvidia-smi"):
        return []
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True,
                         capture_output=True, stdin=subprocess.DEVNULL).stdout
    return [int(x) for x in out.split() if x.isdigit()]

def pid_alive(pid) -> bool:
    """True while a process with this PID exists. Linux /proc only (the drill runs on Colab); False elsewhere,
    so nothing ever signals a PID just to probe it."""
    return Path(f"/proc/{int(pid)}").exists()

def proc_cmdline(pid) -> str:
    """The command line of a process in this PID namespace ('' when it is gone or not visible)."""
    try:
        return Path(f"/proc/{int(pid)}/cmdline").read_bytes().replace(b"\0", b" ").decode("utf-8", "replace")
    except OSError:
        return ""

def drill_orphans(gpu_pids_left, child_pid, marker="LAYA_POC_KILL_DRILL"):
    """(ours, others): ours are the kill drill's own GPU holders, recognised by the marker in their command
    line, and are safe to kill; never this kernel. nvidia-smi may report host-namespace PIDs, so any other
    PID is only reported: the same number can be an unrelated process in this container."""
    me = os.getpid()
    candidates = {int(p) for p in gpu_pids_left} | {int(child_pid)}
    ours = sorted(p for p in candidates if p != me and marker in proc_cmdline(p))
    others = sorted(p for p in {int(p) for p in gpu_pids_left} if p != me and p not in ours)
    return ours, others

def read_events(path) -> list:
    """The JSON events of a run log; unparsable lines (a SIGKILL can leave half a line) are skipped."""
    path, events = Path(path), []
    for line in path.read_text(encoding="utf-8").splitlines() if path.exists() else []:
        try:
            event = json.loads(line)
        except ValueError:
            continue
        if isinstance(event, dict):
            events.append(event)
    return events

def crash_problem(exit_file, log_file, kill_at=None):
    """Why E1's forced crash has not happened as planned, or None when it has. Unlike the smoke drill, any
    number of resumes may follow it (after a Colab disconnect Step 9 simply resumes again), so the log only
    has to contain the injected crash, at micro-step kill_at when given."""
    exit_file = Path(exit_file)
    if not exit_file.exists():
        return "no crash_exit.json: the crash run (Step 8) has not finished"
    rc = json.loads(exit_file.read_text(encoding="utf-8")).get("returncode")
    if rc not in (-9, 137):
        return f"the crash run exited with {rc}, not -9/137 (hard kill)"
    crashes = [e for e in read_events(log_file) if e.get("event") == "crash_injected"]
    if not crashes:
        return f"{Path(log_file).name} has no crash_injected event"
    step = crashes[0].get("micro_step")
    if kill_at is not None and step != kill_at:
        return f"the crash was injected at micro-step {step}, not at the planned {kill_at}"
    return None

def next_log(logs, stem) -> Path:
    """logs/<stem>.log, or <stem>_2.log, <stem>_3.log, ... when earlier attempts exist: a resume after a
    disconnect keeps the logs of the attempts before it."""
    logs, n = Path(logs), 1
    while (logs / (f"{stem}.log" if n == 1 else f"{stem}_{n}.log")).exists():
        n += 1
    return logs / (f"{stem}.log" if n == 1 else f"{stem}_{n}.log")

def trainer_pids(module="laya_poc.train_single") -> list:
    """PIDs of running trainers other than this kernel (Linux /proc; [] elsewhere). A trainer started before a
    kernel restart can outlive the kernel for a while: a second one must never start on the same run dir."""
    proc = Path("/proc")
    if not proc.is_dir():
        return []
    pids = [int(p.name) for p in proc.iterdir() if p.name.isdigit() and int(p.name) != os.getpid()]
    return sorted(pid for pid in pids if module in proc_cmdline(pid))

def fnum(value, nd=4) -> str:
    """A metric for a one-line summary: rounded when numeric, else as is (None when absent)."""
    return f"{value:.{nd}f}" if isinstance(value, float) else str(value)

def dig(obj, *keys):
    """obj[k1][k2]... or None when any level is missing (to print optional report fields)."""
    for key in keys:
        if not isinstance(obj, dict) or key not in obj:
            return None
        obj = obj[key]
    return obj

def elapsed(t0) -> str:
    """Wall-clock time since t0, for the one-line duration every step prints at its end."""
    seconds = time.time() - t0
    return f"{seconds:.0f} s" if seconds < 120 else f"{seconds / 60:.1f} min"

def card_note(env_json, card):
    """A warning when Step 4 found another GPU than the card profile CARD trains with, else None (also for the
    local CPU dry run, which trains on the CPU profile by design)."""
    path = Path(env_json)
    env = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    if card == "CPU" or env.get("card") == card:
        return None
    gpu = env.get("gpu_name") or "no GPU"
    return (f"WARNING: this runtime has {gpu} (card profile {env.get('card')}) but every run trains with "
            f"--card {card}. Timings and the plan assume a G4. Set CARD in Step 1 to this card's profile "
            "(config train.micro_batch) before the first training step, or Remove Server and ask for a G4.")

def busy_pids() -> list:
    """Matrix or trainer processes still running (e.g. from before a kernel restart): a second one must never
    start on the same run directories."""
    return sorted(set(trainer_pids("laya_poc.matrix")) | set(trainer_pids("laya_poc.train_single")))

def show_runs(csv_path, arm=None, cols=("run_name", "val_macro_f1", "test_id_macro_f1", "T", "order_invariance",
                                        "run_seconds")) -> list:
    """Print one line per finished run in results/runs.csv (only `arm`'s when given); return those rows."""
    import csv

    def cell(col, value):
        try:
            return value if "." not in value else f"{float(value):.{0 if col.endswith('seconds') else 4}f}"
        except ValueError:
            return value

    path = Path(csv_path)
    if not path.exists():
        print(f"no {path} yet: no run has finished")
        return []
    with open(path, encoding="utf-8", newline="") as fh:
        rows = [r for r in csv.DictReader(fh) if arm is None or r.get("arm") == arm]
    for row in rows:
        print("  " + " | ".join(f"{c} {cell(c, row.get(c) or '')}" for c in cols))
    return rows

def baseline_pids() -> list:
    """The matrix or a baseline process still running (e.g. from before a kernel restart): a second matrix must
    never start on the same run directories."""
    modules = ("laya_poc.matrix", "laya_poc.baseline_runs", "laya_poc.small_encoder", "laya_poc.llm_baseline")
    return sorted({pid for module in modules for pid in trainer_pids(module)})

def p4_card_note(env_json, card):
    """A warning when Step 4 found another GPU than CARD, else None (also for the local CPU dry run)."""
    path = Path(env_json)
    env = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    if card == "CPU" or env.get("card") == card:
        return None
    gpu = env.get("gpu_name") or "no GPU"
    return (f"WARNING: this runtime has {gpu} (card profile {env.get('card')}), not a {card}. B1 and B3 run on the "
            "CPU anyway; B4 and B5 still run, slower than the estimates, and B5 (Qwen3-4B, about 8 GB of weights) "
            "needs a GPU with at least 16 GB. CARD is recorded with every run (runs.csv): set it in Step 1 to "
            "this card's profile, or Remove Server and ask for a G4.")

def phase4_verdict(report_json) -> str:
    """One line with the Phase 4 exit check of a matrix report JSON. The report's Phase 3 exit check counts the
    Laya runs, which are not part of a Phase 4 session: it FAILs there by design and is not this step's verdict."""
    path = Path(report_json)
    ex = (json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}).get("phase4_exit_check") or {}
    if not ex:
        return f"Phase 4 exit check: not found ({path.name} missing or without it)"
    skipped = ex.get("skipped_optional") or []
    note = f"; optional, not run: {', '.join(skipped)}" if skipped else ""
    return (f"Phase 4 exit check: {ex.get('verdict')} ({ex.get('complete')}/{ex.get('total')} baseline runs "
            f"complete{note}). The Phase 3 exit check above counts the Laya runs of the Phase 3 session, not these.")


SENTINEL = WORK / "sentinel.json"
if SENTINEL.exists():
    print("sentinel from", json.loads(SENTINEL.read_text(encoding="utf-8"))["created"], "found: /content survived")
else:
    write_json(SENTINEL, {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "pid": os.getpid()})
    print("sentinel created: first Step 1 on this server (if Step 1 ran here before, /content was wiped)")
print("HF token: restored from the token file" if restore_hf_token() else "HF token: not set yet (Step 3)")

import base64, hashlib, io, pathlib, zipfile
def _unpack(b64, sha256, dest):
    raw = base64.b64decode(b64)
    assert hashlib.sha256(raw).hexdigest() == sha256, "bundle checksum mismatch"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for n in zf.namelist():
            assert dest.resolve() in (dest / n).resolve().parents, f"unsafe path {n}"
        zf.extractall(dest)
        return zf.namelist()
BUNDLE_SHA256 = "cf73c341309ed89673b8e1f7ab84bfb570e9da8b0e90383137fdb2ff3bf7f1d2"
BUNDLE_B64 = "..."
print(f"unpacked {len(_unpack(BUNDLE_B64, BUNDLE_SHA256, WORK))} files into {WORK}; bundle sha256 {BUNDLE_SHA256[:16]}")


### Step 2 - Install
Plain `laya` at the pinned commit, DuckDB pinned to 1.5.5 (the frozen splits depend on its hash()), this project. Fails if pip changed Colab's torch, transformers, protobuf or numpy. 1-3 min.

In [ ]:
# Step 2 - Install
import importlib, importlib.metadata as md

LAYA_COMMIT = "4066d5d5fbf08b66c6757ddeedbd797bd7655bc0"
PIP = [sys.executable, "-m", "pip", "install", "-q", "--no-input", "--progress-bar", "off"]


def dist_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


def direct_url(name):  # PEP 610 metadata: what pip actually installed, and from where
    try:
        return json.loads(md.distribution(name).read_text("direct_url.json") or "{}")
    except md.PackageNotFoundError:
        return {}


before = {p: dist_version(p) for p in ("torch", "transformers", "protobuf", "numpy")}  # Colab's own builds
if direct_url("laya").get("vcs_info", {}).get("commit_id") != LAYA_COMMIT:  # no [onnx] extra: it pulls protobuf 6+
    run_logged(PIP + [f"laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}"], LOGS / "02_install_laya.log")
DUCKDB = "1.5.5"  # config data.duckdb_version: hash()-based sampling must match the frozen build exactly
if dist_version("duckdb") != DUCKDB:
    run_logged(PIP + [f"duckdb=={DUCKDB}"], LOGS / "02_install_duckdb.log")
if not direct_url("laya-poc").get("dir_info", {}).get("editable"):
    run_logged(PIP + ["--no-deps", "-e", str(WORK)], LOGS / "02_install_project.log")
importlib.invalidate_caches()
changed = {p: f"{before[p]} -> {dist_version(p)}" for p in before if dist_version(p) != before[p]}
if changed:
    raise RuntimeError(f"pip changed {changed}; run Colab: Remove Server and start again")
got = direct_url("laya").get("vcs_info", {}).get("commit_id")
if got != LAYA_COMMIT:
    raise RuntimeError(f"installed laya commit {got} != pinned {LAYA_COMMIT}")
if dist_version("duckdb") != DUCKDB:
    raise RuntimeError(f"duckdb {dist_version('duckdb')} != pinned {DUCKDB}; see {LOGS / '02_install_duckdb.log'}")
print({p: dist_version(p) for p in ("torch", "transformers", "huggingface_hub", "duckdb", "laya", "laya-poc")},
      "laya commit", got[:12])


### Step 3 - Hugging Face token
Masked input box at the top of VS Code; never printed. Unattended: `python tools/build_p4_notebook.py --with-token` and open the gitignored `notebooks/phase4_baselines.local.ipynb`.

In [ ]:
# Step 3 - Hugging Face token
import re
from huggingface_hub import login

PRESET_TOKEN = ""  # filled only in the gitignored phase4_baselines.local.ipynb (build_p4_notebook.py --with-token)
ASK_AGAIN = False  # True: ignore the saved token (e.g. one without access to the gated dataset) and ask for a new one
NEW_TOKEN = "set ASK_AGAIN = True at the top of this cell and re-run it to enter a new token"
restore_hf_token()
token = os.environ.pop("HF_TOKEN", "").strip()
if PRESET_TOKEN and not ASK_AGAIN:  # unattended run: no prompt
    token = PRESET_TOKEN.strip()
del PRESET_TOKEN
if ASK_AGAIN or not token:  # VS Code shows a masked input box at the top of the window
    token = getpass.getpass("Hugging Face token (read access to foursquare/fsq-os-places; hidden): ").strip()
if not token:
    raise RuntimeError("no token entered (Esc/cancel sends an empty string); re-run this cell and paste it")
if not re.fullmatch(r"hf_[A-Za-z0-9]{20,}", token):  # never echo it: a line break would leak the rest in errors
    del token
    raise RuntimeError("that does not look like a Hugging Face token (hidden characters or line breaks?): "
                       + NEW_TOKEN)
try:
    login(token=token)  # validates with whoami and saves the token file, so a restart does not re-prompt
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    if status in (401, 403):
        raise RuntimeError(f"the Hub rejected the token (HTTP {status}): {NEW_TOKEN}") from None
    text = str(exc).replace(token, "***")[:200]  # the exact token first, then anything token-like
    detail = re.sub(r"hf_\w+", "hf_***", type(exc).__name__ + (f", HTTP {status}" if status else "") + f": {text}")
    raise RuntimeError(f"could not reach the Hub to check the token ({detail}); check the network and re-run this "
                       f"cell, or {NEW_TOKEN}") from None
os.environ["HF_TOKEN"] = token  # inherited by the CLIs (DuckDB reads the gated dataset with it)
del token
print("HF token OK (hidden)")


### Step 4 - Environment check
GPU, driver, VRAM, disk, versions, Laya commit -> `runs/p4/env.json`. Warns when the GPU is not a G4. ~30 s.

In [ ]:
# Step 4 - Environment check
T0 = time.time()
if shutil.which("nvidia-smi"):
    sh(["nvidia-smi"], check=False)
env_json = RUNS / "env.json"
env_json.unlink(missing_ok=True)  # a stale file must not explain this run's failure
cmd = [sys.executable, "-m", "laya_poc.env_check", "--out", str(RUNS / "env.json"), "--require-gpu",
       "--expect-card", CARD]
if run_logged(cmd, LOGS / "04_env.log", expect_returncode=None):  # each error in env.json names its fix
    errors = json.loads(env_json.read_text(encoding="utf-8")).get("errors") if env_json.exists() else None
    detail = "; ".join(errors) if errors else f"no env.json written; see {LOGS / '04_env.log'}"
    raise RuntimeError(f"environment check failed: {detail} (runbook: When something fails)")
env = show_json(env_json, ["gpu_name", "capability", "has_sm75", "driver", "vram_total_gb", "disk_free_gb", "torch", "transformers", "laya_commit", "card", "warnings"])
note = p4_card_note(env_json, CARD)
if note:
    print(note)
print("Step 4 - Environment check:", elapsed(T0))


### Step 5 - Full data (frozen)
The E1 build, identical: FULL extraction over `hf://`, trap candidates held out of every split, JSONL, then `--verify-frozen` (the fingerprint must equal `data.frozen_manifest`, else the step fails). 3-20 min.

In [ ]:
# Step 5 - Full data (frozen)
T0 = time.time()
FROZEN = 'docs/results/data_fingerprint_2026-09-15.json'  # config data.frozen_manifest: --verify-frozen compares the build with it
if not already_done(DATA / "build_ok.json"):
    (DATA / "build_ok.json").unlink(missing_ok=True)
    cmd = [sys.executable, "-m", "laya_poc.build_data", "--out", str(DATA), "--verify-frozen"]
    run_logged(cmd, LOGS / "05_data.log",
               hint="HTTP 429 Too Many Requests is a Hugging Face rate limit: wait ~5 min and re-run this step; 401/403: accept the terms of the gated dataset foursquare/fsq-os-places on the Hub; for a different token set ASK_AGAIN = True in Step 3 and re-run it; then re-run this step; if the fingerprint differs from the frozen manifest, this build is not the frozen E1 data: compare the versions in data_report.json with the manifest and do not train on it (runbook)")
    write_json(DATA / "build_ok.json", {"frozen_manifest": FROZEN, "time": time.strftime("%Y-%m-%d %H:%M")})
data_report = show_json(DATA / "data_report.json", ["split_sizes", "headline_rule", "traps", "fingerprint_sha256", "frozen_check", "warnings"])
print("Step 5 - Full data (frozen):", elapsed(T0))


### Step 6 - Data variants
From the frozen `data/`: `data_c7/`, `data_eval/` - the 7-class labels (B1/B3 c7) and the trap candidates as eval rows for c10 and c7. No learning-curve subsets. A partial directory from an interrupted run is deleted and rebuilt. 2-5 min.

In [ ]:
# Step 6 - Data variants
T0 = time.time()
if not already_done(WORK / "data_c7" / "build_ok.json"):
    shutil.rmtree(WORK / "data_c7", ignore_errors=True)  # a partial build, if any
    cmd = [sys.executable, "-m", "laya_poc.variants", "c7", "--data-dir", str(DATA),
           "--out", str(WORK / "data_c7")]
    run_logged(cmd, LOGS / "06_variant_data_c7.log")
    write_json(WORK / "data_c7" / "build_ok.json", {"time": time.strftime("%Y-%m-%d %H:%M")})
if not already_done(WORK / "data_eval" / "build_ok.json"):
    shutil.rmtree(WORK / "data_eval", ignore_errors=True)  # a partial build, if any
    cmd = [sys.executable, "-m", "laya_poc.variants", "traps", "--data-dir", str(DATA),
           "--out", str(WORK / "data_eval")]
    run_logged(cmd, LOGS / "06_variant_data_eval.log")
    write_json(WORK / "data_eval" / "build_ok.json", {"time": time.strftime("%Y-%m-%d %H:%M")})
for d in sorted(p for p in WORK.glob("data_*") if (p / "build_ok.json").exists()):
    print(f"{d.name}: {len(list(d.glob('*.jsonl')))} JSONL files")
print("Step 6 - Data variants:", elapsed(T0))


### Step 7 - B1: majority class + class prior (CPU), c10/c7
`fsq-c10-B1-majority`, `fsq-c10-B1-prior`, `fsq-c7-B1-majority`, `fsq-c7-B1-prior`. Fit on the clean train split on the CPU, T on `val`, every split evaluated with predictions (the no-evidence gate as in `evaluate`). Finished runs are skipped. ~2-3 min.

In [ ]:
# Step 7 - B1: majority class + class prior (CPU), c10/c7
T0 = time.time()
ARM = "B1"
busy = baseline_pids()
if busy:  # e.g. the matrix from before a kernel restart: two must never write into the same run directories
    raise RuntimeError(f"a matrix or baseline process is still running (pid {busy}): wait for it to stop, or stop "
                       "it with os.kill(pid, 9), then re-run this step")
log = next_log(LOGS, "07_B1")  # one log per attempt: a re-run keeps the earlier ones
cmd = [sys.executable, "-m", "laya_poc.matrix", "run", "--work", str(WORK), "--data-root", str(WORK),
       "--only", "B1", "--device", "cuda", "--card", CARD]
rc = run_logged(cmd, log, expect_returncode=None)  # a failed run does not stop the arm's other runs
print(f"{ARM}: finished runs in results/runs.csv")
rows = show_runs(RESULTS / "runs.csv", ARM)
if rc:
    raise RuntimeError(f"{ARM}: matrix run exited with {rc}; the 'matrix:' lines in {log} name each failed run and "
                       "its stage log. " + "Fix the cause, then re-run this step: finished runs are skipped and an unfinished run starts again (runbook: When something fails)")
print("Step 7 - B1:", elapsed(T0))


### Step 8 - B3: char TF-IDF + logistic regression, C and T on val (CPU), c10/c7
`fsq-c10-B3-tfidf_lr`, `fsq-c7-B3-tfidf_lr`. Fit on the clean train split on the CPU, T on `val`, every split evaluated with predictions (the no-evidence gate as in `evaluate`). Finished runs are skipped. ~3-5 min.

In [ ]:
# Step 8 - B3: char TF-IDF + logistic regression, C and T on val (CPU), c10/c7
T0 = time.time()
ARM = "B3"
busy = baseline_pids()
if busy:  # e.g. the matrix from before a kernel restart: two must never write into the same run directories
    raise RuntimeError(f"a matrix or baseline process is still running (pid {busy}): wait for it to stop, or stop "
                       "it with os.kill(pid, 9), then re-run this step")
log = next_log(LOGS, "08_B3")  # one log per attempt: a re-run keeps the earlier ones
cmd = [sys.executable, "-m", "laya_poc.matrix", "run", "--work", str(WORK), "--data-root", str(WORK),
       "--only", "B3", "--device", "cuda", "--card", CARD]
rc = run_logged(cmd, log, expect_returncode=None)  # a failed run does not stop the arm's other runs
print(f"{ARM}: finished runs in results/runs.csv")
rows = show_runs(RESULTS / "runs.csv", ARM)
if rc:
    raise RuntimeError(f"{ARM}: matrix run exited with {rc}; the 'matrix:' lines in {log} name each failed run and "
                       "its stage log. " + "Fix the cause, then re-run this step: finished runs are skipped and an unfinished run starts again (runbook: When something fails)")
print("Step 8 - B3:", elapsed(T0))


### Step 9 - B4: fine-tuned small encoder, `modernbert_base` + `mmbert_small`, c10, seeds 11, 22, 33
`fsq-c10-B4-modernbert_base-s11`, `fsq-c10-B4-modernbert_base-s22`, `fsq-c10-B4-modernbert_base-s33`, `fsq-c10-B4-mmbert_small-s11`, `fsq-c10-B4-mmbert_small-s22`, `fsq-c10-B4-mmbert_small-s33`. Each run: fine-tune on the augmented `train_e*` epochs (soft targets, fp16 on CUDA), keep the best val epoch, fit T on `val`, evaluate every split with predictions, order invariance on `test_id`, then keep only `train/best/` for this session. Finished runs are skipped. ~20-35 min.

In [ ]:
# Step 9 - B4: fine-tuned small encoder, `modernbert_base` + `mmbert_small`, c10, seeds 11, 22, 33
T0 = time.time()
ARM = "B4"
busy = baseline_pids()
if busy:  # e.g. the matrix from before a kernel restart: two must never write into the same run directories
    raise RuntimeError(f"a matrix or baseline process is still running (pid {busy}): wait for it to stop, or stop "
                       "it with os.kill(pid, 9), then re-run this step")
log = next_log(LOGS, "09_B4")  # one log per attempt: a re-run keeps the earlier ones
cmd = [sys.executable, "-m", "laya_poc.matrix", "run", "--work", str(WORK), "--data-root", str(WORK),
       "--only", "B4", "--device", "cuda", "--card", CARD]
rc = run_logged(cmd, log, expect_returncode=None)  # a failed run does not stop the arm's other runs
print(f"{ARM}: finished runs in results/runs.csv")
rows = show_runs(RESULTS / "runs.csv", ARM)
if rc:
    raise RuntimeError(f"{ARM}: matrix run exited with {rc}; the 'matrix:' lines in {log} name each failed run and "
                       "its stage log. " + "Fix the cause, then re-run this step: finished runs are skipped and an unfinished run starts again (runbook: When something fails)")
print("Step 9 - B4:", elapsed(T0))


### Step 10 - B5: zero-shot LLM reference, scored subsets, `qwen3_4b`, c10, optional (flag `RUN_B5`)
`fsq-c10-B5-qwen3_4b`. The pinned model scores every option key (teacher-forced log-probabilities, no generation) on subsets: the first 500 labelled `val` rows (T), up to 2000 rows of `test_id` and each OOD pool, all of `stripped_test` and the trap candidates. Optional: `RUN_B5 = False` in the cell skips it (recorded as optional, disabled; not an exit-check failure). Finished runs are skipped. ~10-20 min.

In [ ]:
# Step 10 - B5: zero-shot LLM reference, scored subsets, `qwen3_4b`, c10, optional (flag `RUN_B5`)
T0 = time.time()
RUN_B5 = True  # False: skip this optional reference; the matrix records it as "optional, disabled"
ARM = "B5"
busy = baseline_pids()
if busy:  # e.g. the matrix from before a kernel restart: two must never write into the same run directories
    raise RuntimeError(f"a matrix or baseline process is still running (pid {busy}): wait for it to stop, or stop "
                       "it with os.kill(pid, 9), then re-run this step")
log = next_log(LOGS, "10_B5")  # one log per attempt: a re-run keeps the earlier ones
cmd = [sys.executable, "-m", "laya_poc.matrix", "run", "--work", str(WORK), "--data-root", str(WORK),
       "--only", "B5", "--device", "cuda", "--card", CARD]
if not RUN_B5:
    cmd.append("--skip-optional")
rc = run_logged(cmd, log, expect_returncode=None)  # a failed run does not stop the arm's other runs
print(f"{ARM}: finished runs in results/runs.csv")
rows = show_runs(RESULTS / "runs.csv", ARM)
if rc:
    raise RuntimeError(f"{ARM}: matrix run exited with {rc}; the 'matrix:' lines in {log} name each failed run and "
                       "its stage log. " + "Fix the cause, then re-run this step: finished runs are skipped and an unfinished run starts again (runbook: When something fails)")
print("Step 10 - B5:", elapsed(T0))


### Step 11 - Report and the Phase 4 exit check
`matrix report` over `runs/p4`: every group (mean ± range over seeds) with macro-F1 on val, test_id and each OOD pool, ID→OOD gaps, ECE pre→post, trap-candidate accuracy (unannotated candidates), stripped false-confident rate, order invariance, and the Phase 4 exit check -> `results/phase4_report.md` (+ `.json`, `runs.csv`); the step prints the Phase 4 verdict in one line. The report's Phase 3 exit check reads NOT RUN here (the Laya runs are in the Phase 3 archive), and decision criterion 1 needs them too: merge the archives locally (runbook). Seconds.

In [ ]:
# Step 11 - Report and the Phase 4 exit check
T0 = time.time()
cmd = [sys.executable, "-m", "laya_poc.matrix", "report", "--work", str(WORK), "--runs-root", str(RUNS),
       "--out", str(WORK / "results" / "phase4_report.md")]
rc = run_logged(cmd, LOGS / "11_report.log", expect_returncode=None)
if (RESULTS / "phase4_report.md").exists():
    show_markdown(RESULTS / "phase4_report.md")
rows = show_runs(RESULTS / "runs.csv")
print(phase4_verdict(RESULTS / "phase4_report.json"))
if rc:  # shown first: the report explains a failed exit check better than the exit code
    raise RuntimeError(f"matrix report exited with {rc}; full log: {LOGS / '11_report.log'}")
print("Step 11 - Report and the Phase 4 exit check:", elapsed(T0))


### Step 12 - Archive
---
Zip logs, JSON, per-row predictions, `results/` and the FSQ NOTICE (no weights, no FSQ rows) for **Download...**. Seconds.

In [ ]:
# Step 12 - Archive
MAKE_ARCHIVE = True  # logs, JSON, per-row predictions (ids, probabilities), results/: no weights, no FSQ rows
ARCHIVE = WORK / "phase4_artifacts.zip"
if MAKE_ARCHIVE:
    small, heavy = {".json", ".jsonl", ".md", ".log", ".yaml", ".csv"}, {"ckpt", "best", "best.partial", "best.prev",
                                                                         "final"}
    keep = [p for top in (RUNS, RESULTS) if top.exists() for p in sorted(top.rglob("*"))
            if p.is_file() and p.suffix in small and not heavy & set(p.relative_to(top).parts)]
    keep += [p for p in (DATA / "data_report.json", DATA / "SHA256SUMS", DATA / "NOTICE_FSQ.txt") if p.exists()]
    keep += sorted(WORK.glob("data_*/variant.json"))  # hashes and counts only
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in keep:
            zf.write(p, p.relative_to(WORK).as_posix())
    print(f"{ARCHIVE}: {len(keep)} files, {ARCHIVE.stat().st_size / 1e6:.1f} MB; Colab view > Contents > "
          "right-click > Download...")


### Finish
- The Phase 4 exit check is at the end of Step 11; every run's metrics are in `results/runs.csv`.
- Download `phase4_artifacts.zip`: Colab view in the activity bar > Contents > right-click > **Download...**
- Locally, unzip it next to the Phase 3 archive and run the merged report for decision criterion 1 (early read):
  `python -m laya_poc.matrix report --runs-root runs/p3_colab/runs/p3 --runs-root runs/p4_colab/runs/p4 --archived` (runbook).
- Next, Phase 5: the full metric suite (bootstrap CIs, abstention, stability) from every run's saved predictions,
  trap accuracy once annotation lands, ONNX and the CPU benchmark, and the decision rule.
- Run **Colab: Remove Server** when you are done. It stops the compute-unit spend and deletes `/content`.
